# Charger les modules et configurer le chemin des données
Définir DATA_DIR (ex: "data"), vérifier l’existence du dossier, et importer les dépendances principales (pandas, numpy). Ajouter une cellule VS Code pour afficher le répertoire courant et aider au debug de chemins.

In [16]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np

# Define the data directory
DATA_DIR = "data"

# Check if the data directory exists
if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"The data directory '{DATA_DIR}' does not exist. Please verify the path.")

# Display the current working directory (useful for debugging paths in VS Code)
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\Dell\Documents\M1\C++\revision\DS\projet\Data_science


# Importer `read_data.py` (chargement des JSON)
Dans le notebook, importer les fonctions `load_data`, `load_data_frames`, `load_project_data` depuis read_data. Exécuter `load_project_data(DATA_DIR)` et afficher `data.keys()` ainsi que quelques `head()` pour valider le chargement.

In [17]:
# Import functions from read_data.py
from read_data import load_data, load_data_frames, load_project_data

# Load all project data using the provided data directory
data = load_project_data(DATA_DIR)

# Display the keys of the loaded data dictionary
print("Loaded datasets:", data.keys())

# Display the first few rows of each dataset to validate the loading process
for key, df in data.items():
    print(f"Dataset: {key}")
    print(df.head())  # Display the first few rows of the dataset
    print("-" * 40)

Loaded datasets: dict_keys(['docs', 'qgts_train', 'queries_test', 'queries_train'])
Dataset: docs
                                            id  \
0  6970cbf2-ffff-4c7b-b73d-52524000c232_145427   
1   667220d0-66be-4059-ae60-b9df15d285f7_77850   
2   e5608645-f23d-4cf4-bda6-46071510ebb0_84546   
3  367fcce1-2ed8-4a95-81e4-77e30c1a37eb_154499   
4   75fcda1e-9b8e-4d95-bbd9-770dd6f011cc_41849   

                                                text  \
0  I try to install Complete MiKTeX 2.9 But there...   
1  I'm trying to get a launcher working for the W...   
2  Its unclear as to whether this trait means tha...   
3  I am writing a thesis with specific margin req...   
4  I always see job positions for web companies f...   

                                               title  \
0         MikTex Download Failure - toptesi.tar.lzma   
1  Launcher for a Python program that requires ex...   
2  How does the "Double damage in combat" trait w...   
3  Compiling with "latex" instead of "pd

# Importer `processing_data.py` (prétraitement + colonne `content`)
Importer `prepare_data` puis appliquer le prétraitement sur `data["docs"]` et `data["queries_train"]`. Vérifier que la colonne `content` est créée, afficher `[['id','content']].head()` et contrôler les NaN.

In [18]:
# Import the `prepare_data` function from `processing_data.py`
from processing_data import prepare_data

# Apply preprocessing to create the "content" column for documents and queries
docs_df = prepare_data(data["docs"])
queries_train_df = prepare_data(data["queries_train"])

# Verify that the "content" column is created and display the first few rows
print("Documents DataFrame (with 'content'):")
print(docs_df[['id', 'content']].head())

print("\nQueries DataFrame (with 'content'):")
print(queries_train_df[['id', 'content']].head())

# Check for NaN values in the "content" column
print("\nNaN values in 'content' column (Documents):", docs_df['content'].isna().sum())
print("NaN values in 'content' column (Queries):", queries_train_df['content'].isna().sum())

Documents DataFrame (with 'content'):
                                            id  \
0  6970cbf2-ffff-4c7b-b73d-52524000c232_145427   
1   667220d0-66be-4059-ae60-b9df15d285f7_77850   
2   e5608645-f23d-4cf4-bda6-46071510ebb0_84546   
3  367fcce1-2ed8-4a95-81e4-77e30c1a37eb_154499   
4   75fcda1e-9b8e-4d95-bbd9-770dd6f011cc_41849   

                                             content  
0  miktex download failure - toptesi.tar.lzma i t...  
1  launcher for a python program that requires ex...  
2  how does the "double damage in combat" trait w...  
3  compiling with "latex" instead of "pdflatex" c...  
4  machine learning web jobs i always see job pos...  

Queries DataFrame (with 'content'):
                                            id  \
0   961c4349-8cf1-4ef1-89cc-24d20bb9d000_67878   
1     4008ed78-e66e-4d89-9c3b-c79bd1cf6fc9_366   
2   d5a95b09-e8ea-44dd-993d-347ed418e1f1_15138   
3  3e66798a-b7fd-41b5-8bc0-33b3d7ce2aca_177487   
4  f5f944d2-277a-481d-ab09-612890402ded_1374

# Importer `methodes.py` (TF‑IDF + BM25)
Importer `tfidf_retrieve` et `bm25_retrieve`. Vérifier que les dépendances sont disponibles (scikit-learn, rank_bm25). Ajouter une cellule qui installe/valide les packages si nécessaire (dans l’environnement courant).

In [19]:
# Import retrieval methods from `methodes.py`
import sys

try:
    import sklearn  # noqa: F401
    from rank_bm25 import BM25Plus  # noqa: F401
    print("All required packages are installed (current kernel).")
except ImportError:
    print("Some required packages are missing in this kernel. Installing them now...")
    !"{sys.executable}" -m pip install -U pip
    !"{sys.executable}" -m pip install scikit-learn rank-bm25

# Import retrieval methods from `methodes.py` (after deps are available)
from methodes import tfidf_retrieve, bm25_retrieve

# Re-import to ensure availability after installation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Plus

# Confirm successful import of methods
print("Methods `tfidf_retrieve` and `bm25_retrieve` imported successfully.")

# Optional: show which Python this notebook kernel uses (helps debugging)
print("Notebook kernel Python:", sys.executable)


All required packages are installed (current kernel).
Methods `tfidf_retrieve` and `bm25_retrieve` imported successfully.
Notebook kernel Python: c:\Python312\python.exe


# Exécuter le pipeline : load → prepare → retrieve (TF‑IDF)
Exécuter `tfidf_retrieve(docs_df, queries_train_df, top_k=10)` puis afficher 1–2 résultats. Ajouter des contrôles: longueur des résultats = nb de queries, format `{query_id, relevant_docs}` et taille de `relevant_docs` = top_k.

In [20]:
# Execute the TF-IDF retrieval pipeline
top_k = 10  # Number of top documents to retrieve per query

topk_indices_tfidf, topk_scores_tfidf, tfidf_results = tfidf_retrieve(
    docs_df, queries_train_df, top_k=top_k
)

# Display 1–2 results for inspection
print("TF-IDF Retrieval Results (1–2 queries):")
for result in tfidf_results[:2]:
    print(result)

# Validate the results
assert len(tfidf_results) == len(queries_train_df), "Mismatch: Number of results != Number of queries"

for result in tfidf_results:
    assert "query_id" in result and "relevant_docs" in result, "Invalid result format"
    assert isinstance(result["relevant_docs"], list), "'relevant_docs' should be a list"
    assert len(result["relevant_docs"]) == top_k, f"'relevant_docs' length != {top_k}"

print("\nTF-IDF retrieval pipeline executed successfully. Results validated.")

TF-IDF Retrieval Results (1–2 queries):
{'query_id': '961c4349-8cf1-4ef1-89cc-24d20bb9d000_67878', 'relevant_docs': ['cbac6e51-ea52-4824-be99-93db1abbe35a_62218', '6260dea3-8a5e-4e28-8c8e-9340f9352887_23121', '9b7f0646-d400-4843-b5d8-df4ce4cd7b27_7240', 'a1fdd2bb-79eb-4dc5-8671-afca8cd3dac3_44409', '020acc87-e0cb-43bc-a38a-b6b2872a154c_7289', '7c092c06-046c-4e35-8769-4e912eec36fc_71751', '9794a532-ce89-4280-9f10-68b8c95aa20d_23563', 'b0ef7a11-3f17-4235-b291-62b1b7afa5ab_37373', '50f16aa6-e9aa-4635-a0d5-b31e0265cc6f_16954', 'be55a7d5-7935-434c-9425-783681823ef9_40574'], 'scores': [0.4905728764886795, 0.4735315964919104, 0.4733667996134766, 0.4703303732191364, 0.4686005068794075, 0.4406677358592113, 0.40186925541093055, 0.39106697476652974, 0.38920425932355, 0.3873097990760041]}
{'query_id': '4008ed78-e66e-4d89-9c3b-c79bd1cf6fc9_366', 'relevant_docs': ['fb4558fa-e0db-4dc6-8bef-28905c358520_55298', '012ba6ce-e657-4285-b29b-4ad830598ef2_70387', '7126a671-70e4-4a06-8246-0435f92bca4b_57876',

# Exécuter le pipeline : load → prepare → retrieve (BM25)
Sous-échantillonner (ex: 5000 docs, 3 queries) pour accélérer, puis exécuter `bm25_retrieve(small_docs, small_queries, top_k=5)`. Afficher les résultats et vérifier l’ordre décroissant via les indices (sanity check).

In [21]:
# ===== BM25 retrieval sur tout le dataset =====

# IMPORTANT : utiliser toutes les données
docs_for_bm25 = docs_df
queries_for_bm25 = queries_train_df

top_k = 10  # ou 5 si ton évaluation est Precision@5

# bm25_retrieve retourne (indices, scores, results)
topk_indices_bm25, topk_scores_bm25, bm25_results = bm25_retrieve(
    docs_for_bm25,
    queries_for_bm25,
    top_k=top_k
)

print("BM25 Retrieval Results:")
print("Nombre total de queries traitées:", len(bm25_results))

# afficher un exemple
if bm25_results:
    print("\nExemple résultat:")
    print("Query ID:", bm25_results[0]["query_id"])
    print("Top Docs:", bm25_results[0]["relevant_docs"][:5])

# ===== Validation (comme pour TF-IDF) =====

# Vérifier que le nombre de résultats correspond au nombre de queries
assert len(bm25_results) == len(queries_for_bm25), \
    "Mismatch: Number of results != Number of queries"

# Vérifier le format
for result in bm25_results:
    assert "query_id" in result and "relevant_docs" in result, \
        "Invalid result format"
    assert isinstance(result["relevant_docs"], list), \
        "'relevant_docs' should be a list"
    assert len(result["relevant_docs"]) == top_k, \
        f"'relevant_docs' length != {top_k}"

print("\nBM25 retrieval pipeline executed successfully. Results validated.")

BM25 Retrieval Results:
Nombre total de queries traitées: 327

Exemple résultat:
Query ID: 961c4349-8cf1-4ef1-89cc-24d20bb9d000_67878
Top Docs: ['cbac6e51-ea52-4824-be99-93db1abbe35a_62218', '7c092c06-046c-4e35-8769-4e912eec36fc_71751', '9b7f0646-d400-4843-b5d8-df4ce4cd7b27_7240', '3dc818a3-141d-4352-8e54-db21dc85861e_67816', 'be55a7d5-7935-434c-9425-783681823ef9_40574']

BM25 retrieval pipeline executed successfully. Results validated.


In [22]:
# ===== Diagnostic BM25 =====

print("Type bm25_results:", type(bm25_results))
print("Nombre d'entrées bm25_results:", len(bm25_results))

# Affiche les 3 premières entrées (structure)
for i in range(min(3, len(bm25_results))):
    print(f"\n--- bm25_results[{i}] keys ---")
    if isinstance(bm25_results[i], dict):
        print(bm25_results[i].keys())
        # try common fields
        for k in ["query_id", "query", "retrieved_docs", "ranked_docs", "relevant_docs", "docs"]:
            if k in bm25_results[i]:
                v = bm25_results[i][k]
                if isinstance(v, list):
                    print(f"{k}: list(len={len(v)}) sample={v[:3]}")
                else:
                    print(f"{k}: {type(v)} -> {str(v)[:120]}")
    else:
        print("Entrée non dict:", type(bm25_results[i]))

# Vérifie combien d'entrées ont un query_id exploitable
qid_ok = 0
qid_missing = 0
retr_ok = 0
retr_missing = 0

for r in bm25_results:
    qid = r.get("query_id") if isinstance(r, dict) else None
    if qid:
        qid_ok += 1
    else:
        qid_missing += 1

    # Adapte selon ton evaluate_results : généralement il cherche une liste de docs retrouvés
    # Je teste des noms fréquents
    retrieved = None
    if isinstance(r, dict):
        for k in ["retrieved_docs", "ranked_docs", "docs", "predicted_docs", "results"]:
            if k in r and isinstance(r[k], list):
                retrieved = r[k]
                break

    if retrieved is not None:
        retr_ok += 1
    else:
        retr_missing += 1

print("\nRésumé:")
print("  Entrées avec query_id:", qid_ok)
print("  Entrées sans query_id:", qid_missing)
print("  Entrées avec liste docs retrouvés:", retr_ok)
print("  Entrées sans liste docs retrouvés:", retr_missing)

Type bm25_results: <class 'list'>
Nombre d'entrées bm25_results: 327

--- bm25_results[0] keys ---
dict_keys(['query_id', 'relevant_docs', 'scores'])
query_id: <class 'str'> -> 961c4349-8cf1-4ef1-89cc-24d20bb9d000_67878
relevant_docs: list(len=10) sample=['cbac6e51-ea52-4824-be99-93db1abbe35a_62218', '7c092c06-046c-4e35-8769-4e912eec36fc_71751', '9b7f0646-d400-4843-b5d8-df4ce4cd7b27_7240']

--- bm25_results[1] keys ---
dict_keys(['query_id', 'relevant_docs', 'scores'])
query_id: <class 'str'> -> 4008ed78-e66e-4d89-9c3b-c79bd1cf6fc9_366
relevant_docs: list(len=10) sample=['5752fe23-77bd-4909-a413-81f70b062ac1_96458', '6589d0ba-7577-49c5-81c0-cd9ef3582dc5_82336', '1f8673cd-dc13-47ed-901f-82e7b2df92d9_128975']

--- bm25_results[2] keys ---
dict_keys(['query_id', 'relevant_docs', 'scores'])
query_id: <class 'str'> -> d5a95b09-e8ea-44dd-993d-347ed418e1f1_15138
relevant_docs: list(len=10) sample=['fbbe56d1-168d-4429-a9e2-8e98ac8fb86d_17529', '4ad1a51e-68cb-4ac2-982e-7b20b613d949_58347', '89c

# Vérifications rapides (shapes, colonnes, aperçu, exemples de résultats)
Afficher `shape`, `columns`, nombre de documents/queries. Vérifier que `id` existe dans docs/queries. Afficher quelques contenus et quelques listes de `relevant_docs` pour confirmer que les IDs proviennent bien de `docs_df['id']`.

In [23]:
# Verify the shapes of the datasets
print("Shape of Documents DataFrame:", docs_df.shape)
print("Shape of Queries DataFrame:", queries_train_df.shape)

# Verify the columns in the datasets
print("\nColumns in Documents DataFrame:", docs_df.columns.tolist())
print("Columns in Queries DataFrame:", queries_train_df.columns.tolist())

# Check if 'id' column exists in both datasets
assert 'id' in docs_df.columns, "'id' column is missing in Documents DataFrame"
assert 'id' in queries_train_df.columns, "'id' column is missing in Queries DataFrame"
print("\n'id' column exists in both datasets.")

# Display the number of documents and queries
print("\nNumber of documents:", len(docs_df))
print("Number of queries:", len(queries_train_df))

# Display a few rows of the 'content' column for both datasets
print("\nSample 'content' from Documents DataFrame:")
print(docs_df[['id', 'content']].head())

print("\nSample 'content' from Queries DataFrame:")
print(queries_train_df[['id', 'content']].head())

# Display a few results from the retrieval methods
print("\nSample TF-IDF Retrieval Results:")
for result in tfidf_results[:2]:
    print("Query ID:", result["query_id"])
    print("Relevant Docs:", result["relevant_docs"])

print("\nSample BM25 Retrieval Results:")
for result in bm25_results[:2]:
    print("Query ID:", result["query_id"])
    print("Relevant Docs:", result["relevant_docs"])

# Verify that the IDs in 'relevant_docs' exist in the Documents DataFrame
for result in bm25_results[:2]:
    relevant_docs = result["relevant_docs"]
    assert all(doc_id in docs_df['id'].values for doc_id in relevant_docs), "Some IDs in 'relevant_docs' do not exist in Documents DataFrame"
print("\nVerification complete: All IDs in 'relevant_docs' exist in Documents DataFrame.")

Shape of Documents DataFrame: (216041, 6)
Shape of Queries DataFrame: (327, 6)

Columns in Documents DataFrame: ['id', 'text', 'title', 'tags', 'category', 'content']
Columns in Queries DataFrame: ['id', 'text', 'title', 'tags', 'category', 'content']

'id' column exists in both datasets.

Number of documents: 216041
Number of queries: 327

Sample 'content' from Documents DataFrame:


                                            id  \
0  6970cbf2-ffff-4c7b-b73d-52524000c232_145427   
1   667220d0-66be-4059-ae60-b9df15d285f7_77850   
2   e5608645-f23d-4cf4-bda6-46071510ebb0_84546   
3  367fcce1-2ed8-4a95-81e4-77e30c1a37eb_154499   
4   75fcda1e-9b8e-4d95-bbd9-770dd6f011cc_41849   

                                             content  
0  miktex download failure - toptesi.tar.lzma i t...  
1  launcher for a python program that requires ex...  
2  how does the "double damage in combat" trait w...  
3  compiling with "latex" instead of "pdflatex" c...  
4  machine learning web jobs i always see job pos...  

Sample 'content' from Queries DataFrame:
                                            id  \
0   961c4349-8cf1-4ef1-89cc-24d20bb9d000_67878   
1     4008ed78-e66e-4d89-9c3b-c79bd1cf6fc9_366   
2   d5a95b09-e8ea-44dd-993d-347ed418e1f1_15138   
3  3e66798a-b7fd-41b5-8bc0-33b3d7ce2aca_177487   
4  f5f944d2-277a-481d-ab09-612890402ded_137489   

                          

# (Optionnel) Mini benchmarks dans VS Code (temps d’exécution, top_k, sous-échantillonnage)
Ajouter des cellules avec `time`/`timeit` pour comparer TF‑IDF vs BM25, varier `top_k`, et tester l’impact de la taille du corpus. Prévoir une cellule pour écrire les résultats dans un petit DataFrame récapitulatif.

In [24]:
# Import the time module for benchmarking
import time
import pandas as pd

# Define a function to benchmark retrieval methods
def benchmark_retrieval(docs_df, queries_df, top_k_values, corpus_sizes):
    """
    Benchmark TF-IDF and BM25 retrieval methods with varying top_k and corpus sizes.

    Parameters:
        docs_df (pd.DataFrame): DataFrame containing the documents.
        queries_df (pd.DataFrame): DataFrame containing the queries.
        top_k_values (list): List of top_k values to test.
        corpus_sizes (list): List of corpus sizes to test.

    Returns:
        pd.DataFrame: DataFrame summarizing the benchmark results.
    """
    results = []

    for corpus_size in corpus_sizes:
        # Subsample the documents
        small_docs = docs_df.iloc[:corpus_size]

        for top_k in top_k_values:
            # Benchmark TF-IDF
            start_time = time.time()
            tfidf_retrieve(small_docs, queries_df, top_k=top_k)
            tfidf_time = time.time() - start_time

            # Benchmark BM25
            start_time = time.time()
            bm25_retrieve(small_docs, queries_df, top_k=top_k)
            bm25_time = time.time() - start_time

            # Append results
            results.append({
                "Corpus Size": corpus_size,
                "Top K": top_k,
                "TF-IDF Time (s)": tfidf_time,
                "BM25 Time (s)": bm25_time
            })

    return pd.DataFrame(results)

# Define top_k values and corpus sizes for benchmarking
top_k_values = [5, 10, 20]
corpus_sizes = [1000, 5000, 10000]

# Run the benchmark
benchmark_results = benchmark_retrieval(docs_df, queries_train_df, top_k_values, corpus_sizes)

# Display the benchmark results
print("\nBenchmark Results:")
print(benchmark_results)

# Save the benchmark results to a CSV file (optional)
benchmark_results.to_csv("benchmark_results.csv", index=False)


Benchmark Results:
   Corpus Size  Top K  TF-IDF Time (s)  BM25 Time (s)
0         1000      5         0.384111       1.487358
1         1000     10         0.348560       1.455682
2         1000     20         0.363361       1.407659
3         5000      5         1.125032       7.609891
4         5000     10         1.116120       7.393341
5         5000     20         1.088835       7.799058
6        10000      5         2.032766      17.048056
7        10000     10         1.983018      19.345870
8        10000     20         2.056021      17.880227


# Évaluation des résultats de recherche

Dans cette section, on définit des fonctions d'évaluation (Precision@k, Recall@k, MRR@k) et on les applique aux résultats TF‑IDF et BM25.

Les `results` ont le format :

- `query_id` : identifiant de la requête
- `relevant_docs` : liste ordonnée des ids de documents retournés par la méthode

In [25]:
import numpy as np
import pandas as pd

def build_gts_map(gts):
    """
    Construit un dictionnaire {query_id: set(doc_id_relevants)} à partir de gts.
    Compatible avec plusieurs formats possibles :
      - DataFrame avec colonnes: query_id + doc_id (1 ligne = 1 doc relevant)
      - DataFrame avec colonnes: query_id + relevant_docs (liste de doc ids)
      - DataFrame où chaque colonne est un query_id et chaque cellule la structure qgts_train (cas particulier)
      - list[dict] avec {"query_id": ..., "relevant_docs": [...]}
      - dict déjà prêt {query_id: [...] ou set(...)}
      - dict au format qgts_train.json (clé = query_id, valeur["relevant_doc_ids"] = liste de {doc_id, relevance})
    """
    # Cas 1 : dict déjà mapping simple ou dict au format qgts_train
    if isinstance(gts, dict):
        sample_val = next(iter(gts.values())) if gts else None

        # Si les valeurs sont déjà des listes/sets de doc_ids
        if isinstance(sample_val, (list, set, tuple, str)) or sample_val is None:
            return {k: (set(v) if not isinstance(v, set) else v) for k, v in gts.items()}

        # Format qgts_train.json: {qid: { 'relevant_doc_ids': [ {doc_id, relevance}, ... ], ... }}
        if isinstance(sample_val, dict) and "relevant_doc_ids" in sample_val:
            m = {}
            for qid, meta in gts.items():
                rel_list = meta.get("relevant_doc_ids", [])
                m[qid] = set(d["doc_id"] for d in rel_list if "doc_id" in d)
            return m

        # Sinon, dernier recours : essayer de lire une clé 'relevant_docs' ou 'doc_ids',
        # et ignorer les valeurs scalaires non itérables (comme des int).
        m = {}
        for qid, val in gts.items():
            rel = None
            if isinstance(val, dict):
                rel = val.get("relevant_docs") or val.get("doc_ids")
            # Si rien de pertinent trouvé ou si valeur scalaire, on saute cette entrée
            if rel is None:
                continue
            # Si rel est un seul doc_id sous forme de string, on le met dans une liste
            if isinstance(rel, str):
                rel = [rel]
            m[qid] = set(rel)
        return m

    # Cas 2 : list[dict]
    if isinstance(gts, list):
        m = {}
        for item in gts:
            qid = item.get("query_id")
            rel = item.get("relevant_docs") or item.get("doc_ids") or item.get("relevant") or []
            m[qid] = set(rel)
        return m

    # Cas 3 : DataFrame
    if isinstance(gts, pd.DataFrame):
        cols = set(gts.columns)

        # cas 1: (query_id, doc_id) => une ligne par doc relevant
        if {"query_id", "doc_id"}.issubset(cols):
            m = {}
            for qid, grp in gts.groupby("query_id"):
                m[qid] = set(grp["doc_id"].tolist())
            return m

        # cas 2: (query_id, relevant_docs) => liste dans une cellule
        if {"query_id", "relevant_docs"}.issubset(cols):
            return {row["query_id"]: set(row["relevant_docs"]) for _, row in gts.iterrows()}

        # cas 3: DataFrame issu de qgts_train.json: colonnes = query_ids
        # On reconstruit un dict {qid: meta_dict} puis on réutilise la logique dict ci‑dessus.
        try:
            as_dict = {col: gts[col].iloc[0] for col in gts.columns}
            return build_gts_map(as_dict)
        except Exception:
            raise ValueError(f"Format gts DataFrame non reconnu. Colonnes trouvées: {list(gts.columns)}")

    raise TypeError("gts doit être un dict, une list[dict], ou un pandas.DataFrame")


def precision_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for d in retrieved_k if d in relevant_docs)
    return hits / k


def recall_at_k(retrieved_docs, relevant_docs, k):
    if len(relevant_docs) == 0:
        return 0.0
    retrieved_k = retrieved_docs[:k]
    hits = sum(1 for d in retrieved_k if d in relevant_docs)
    return hits / len(relevant_docs)


def mrr_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    for rank, d in enumerate(retrieved_k, start=1):
        if d in relevant_docs:
            return 1.0 / rank
    return 0.0


def evaluate_results(results, gts, k=10, strict=True):
    """
    results: list[dict] contenant au minimum:
        - "query_id"
        - "relevant_docs" (liste ordonnée de docs récupérés)
    gts: mapping/dataset ground truth (voir build_gts_map)

    strict=True : ignore les queries qui n'existent pas dans gts.
    strict=False: considère relevant_docs vide si pas trouvé.
    """
    gts_map = build_gts_map(gts)

    precisions, recalls, mrrs = [], [], []
    missing_in_gts = 0

    for item in results:
        qid = item["query_id"]
        retrieved = item["relevant_docs"]

        if qid not in gts_map:
            missing_in_gts += 1
            if strict:
                continue
            relevant = set()
        else:
            relevant = gts_map[qid]

        precisions.append(precision_at_k(retrieved, relevant, k))
        recalls.append(recall_at_k(retrieved, relevant, k))
        mrrs.append(mrr_at_k(retrieved, relevant, k))

    metrics = {
        f"Precision@{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"Recall@{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"MRR@{k}": float(np.mean(mrrs)) if mrrs else 0.0,
        "num_queries_evaluated": int(len(precisions)),
        "num_missing_in_gts": int(missing_in_gts),
    }
    return metrics

In [27]:
# ===== Construction correcte du ground truth et évaluation =====

import json
import pandas as pd

# 1) Charger le ground truth correctement
if "gts" in data:
    gts_raw = data["gts"]
elif "qgts_train" in data and not isinstance(data["qgts_train"], pd.DataFrame):
    gts_raw = data["qgts_train"]
else:
    # fallback : charger directement depuis fichier JSON
    GT_PATH = "data/qgts_train.json"
    with open(GT_PATH, "r", encoding="utf-8") as f:
        gts_raw = json.load(f)

print("Type du ground truth:", type(gts_raw))

# 2) Construire mapping {query_id: set(doc_ids)}
gts_map = build_gts_map(gts_raw)
print("Nombre de queries dans le ground truth:", len(gts_map))

# 3) Évaluation
k_eval = 10

tfidf_metrics = evaluate_results(
    tfidf_results,
    gts_map,
    k=k_eval,
    strict=True
)

bm25_top_k = len(bm25_results[0]["relevant_docs"]) if bm25_results else 0

bm25_metrics = evaluate_results(
    bm25_results,
    gts_map,
    k=min(k_eval, bm25_top_k),
    strict=True
)

# 4) Affichage
print("\nTF-IDF metrics:")
for key, value in tfidf_metrics.items():
    print(f"  {key}: {value}")

print("\nBM25 metrics:")
for key, value in bm25_metrics.items():
    print(f"  {key}: {value}")

# 5) Diagnostic rapide si problème
if len(gts_map) == 0:
    print("\nERREUR: ground truth vide.")
    if isinstance(gts_raw, dict) and len(gts_raw) > 0:
        sample_key = next(iter(gts_raw))
        print("Exemple clé:", sample_key)
        print("Valeur:", gts_raw[sample_key])

Type du ground truth: <class 'dict'>
Nombre de queries dans le ground truth: 327

TF-IDF metrics:
  Precision@10: 0.06269113149847094
  Recall@10: 0.10734834189928322
  MRR@10: 0.1703388185039561
  num_queries_evaluated: 327
  num_missing_in_gts: 0

BM25 metrics:
  Precision@10: 0.07767584097859327
  Recall@10: 0.12796347204988723
  MRR@10: 0.23449832532401338
  num_queries_evaluated: 327
  num_missing_in_gts: 0
